# 01 — Image Branch: EfficientNet-B4
### Phase 1 of the implementation plan

Goal of this notebook:

1. Load a **pretrained EfficientNet-B4** (ImageNet weights) via `timm`
2. Replace the classifier head with a `Real vs Fake` head and **fine-tune** it (transfer learning — backbone mostly frozen, head + last block trainable)
3. Define a **feature-extractor function** that returns the penultimate-layer embedding (`F_image`) instead of the classification logits — this is what the fusion network will consume later
4. Train/validate on your `data/image/real` and `data/image/fake` folders (full dataset)
5. Save the fine-tuned weights to `checkpoints/efficientnet_b4_image.pt`

Corresponds to: `Image → EfficientNet-B4 → Image Feature Vector` in the project design.

In [1]:
import os, json, random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from PIL import Image
import timm

# Throttle CPU threads to keep CPU cool (<60°C)
torch.set_num_threads(2)

with open("shared_config.json") as f:
    CFG = json.load(f)

PROJECT_ROOT = CFG["project_root"]
IMG_CFG = CFG["image"]
SEED = CFG["seed"]

random.seed(SEED); torch.manual_seed(SEED)

try:
    import torch_directml
    device = torch_directml.device()
    print("Device: DirectML GPU (AMD Radeon RX 580)", device)
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

Device: DirectML GPU (AMD Radeon RX 580) privateuseone:0


## 1. Dataset

Expects:
```
data/image/real/*.jpg|png
data/image/fake/*.jpg|png
```
Uses full dataset (`max_samples_per_class=None`).

In [2]:
IMG_SIZE = IMG_CFG["img_size"]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImageRealFakeDataset(Dataset):
    """Loads data/image/real and data/image/fake."""
    def __init__(self, root, transform, max_samples_per_class=None, n_synthetic=64):
        self.transform = transform
        self.samples = []
        for label, cls in enumerate(["real", "fake"]):
            cls_dir = os.path.join(root, cls)
            if os.path.isdir(cls_dir):
                files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                if max_samples_per_class and len(files) > max_samples_per_class:
                    files = files[:max_samples_per_class]
                for fname in files:
                    self.samples.append((os.path.join(cls_dir, fname), label))

        self.synthetic = len(self.samples) == 0
        if self.synthetic:
            print(f"No real images found under {root} — using {n_synthetic} synthetic samples so the pipeline can be validated end-to-end.")
            self.n_synthetic = n_synthetic

    def __len__(self):
        return self.n_synthetic if self.synthetic else len(self.samples)

    def __getitem__(self, idx):
        if self.synthetic:
            img = Image.fromarray((torch.rand(IMG_SIZE, IMG_SIZE, 3).numpy() * 255).astype("uint8"))
            label = idx % 2
        else:
            path, label = self.samples[idx]
            img = Image.open(path).convert("RGB")
        return self.transform(img), label

full_dataset = ImageRealFakeDataset(os.path.join(PROJECT_ROOT, "data/image"), train_tf, max_samples_per_class=None)
val_len = max(1, int(0.2 * len(full_dataset)))
train_len = len(full_dataset) - val_len
train_ds, val_ds = random_split(full_dataset, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=IMG_CFG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=IMG_CFG["batch_size"], shuffle=False)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Train samples: 8000 | Val samples: 2000


## 2. Model — EfficientNet-B4 with a Real/Fake head

We keep the backbone mostly frozen (transfer learning) and only train the classifier + last block, per the project's feasibility notes.

In [3]:
class EfficientNetB4Detector(nn.Module):
    def __init__(self, backbone_name=IMG_CFG["backbone"], num_classes=2, freeze_backbone=True):
        super().__init__()
        # num_classes=0 -> timm returns pooled *features* instead of classification logits
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            # Unfreeze just the last block(s) so the model can still adapt to deepfake artifacts
            for name, param in self.backbone.named_parameters():
                if any(k in name for k in ["blocks.6", "blocks.5", "conv_head", "bn2"]):
                    param.requires_grad = True

    def extract_features(self, x):
        """Returns F_image — the embedding BEFORE the classifier head. This is what the fusion network will consume later."""
        return self.backbone(x)

    def forward(self, x):
        feats = self.extract_features(x)
        return self.classifier(feats)

model = EfficientNetB4Detector().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"Feature vector dimension (F_image): {model.feature_dim}")

Trainable params: 14,394,426 / 18,008,138 (79.9%)
Feature vector dimension (F_image): 1792


## 3. Train (fine-tune)

Prints batch progress line-by-line.

In [4]:
import sys, time

EPOCHS = 3
LR = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

def run_epoch(epoch, loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    total_items = len(loader.dataset)
    total_batches = len(loader)
    mode = "Train" if train else "Val"
    start_time = time.time()
    
    with torch.set_grad_enabled(train):
        for batch_idx, (imgs, labels) in enumerate(loader, 1):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = imgs.size(0)
            total_loss += loss.item() * batch_size
            correct += (outputs.argmax(1) == labels).sum().item()
            n += batch_size

            elapsed = time.time() - start_time
            speed = n / elapsed if elapsed > 0 else 0
            remaining_sec = (total_items - n) / speed if speed > 0 else 0
            el_m, el_s = divmod(int(elapsed), 60)
            rem_m, rem_s = divmod(int(remaining_sec), 60)
            pct = (n / total_items) * 100
            
            msg = f"\rEpoch {epoch}/{EPOCHS} [{mode}] | Batch {batch_idx}/{total_batches} ({n}/{total_items}, {pct:.1f}%) | Loss: {total_loss/n:.4f} | Acc: {correct/n:.3f} | {speed:.1f} img/s | [{el_m:02d}:{el_s:02d}<{rem_m:02d}:{rem_s:02d}]"
            sys.stdout.write(msg)
            sys.stdout.flush()
            
    sys.stdout.write("\n")
    sys.stdout.flush()
    return total_loss / n, correct / n

print("Starting EfficientNet-B4 Training...", flush=True)
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(epoch, train_loader, train=True)
    val_loss, val_acc = run_epoch(epoch, val_loader, train=False)
    print(f"--> Epoch {epoch}/{EPOCHS} Complete | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.3f}\n", flush=True)


Starting EfficientNet-B4 Training...


c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Epoch 1/3 [Train] | Batch 500/500 (8000/8000, 100.0%) | Loss: 0.2214 | Acc: 0.910 | 6.1 img/s | [21:52<00:00]
Epoch 1/3 [Val] | Batch 125/125 (2000/2000, 100.0%) | Loss: 0.0658 | Acc: 0.978 | 22.2 img/s | [01:30<00:00]
--> Epoch 1/3 Complete | Train Loss: 0.2214, Train Acc: 0.910 | Val Loss: 0.0658, Val Acc: 0.978

Epoch 2/3 [Train] | Batch 500/500 (8000/8000, 100.0%) | Loss: 0.0548 | Acc: 0.980 | 6.4 img/s | [20:56<00:00]
Epoch 2/3 [Val] | Batch 125/125 (2000/2000, 100.0%) | Loss: 0.0465 | Acc: 0.985 | 26.2 img/s | [01:16<00:00]
--> Epoch 2/3 Complete | Train Loss: 0.0548, Train Acc: 0.980 | Val Loss: 0.0465, Val Acc: 0.985

Epoch 3/3 [Train] | Batch 500/500 (8000/8000, 100.0%) | Loss: 0.0306 | Acc: 0.990 | 6.4 img/s | [20:50<00:00]
Epoch 3/3 [Val] | Batch 125/125 (2000/2000, 100.0%) | Loss: 0.0366 | Acc: 0.988 | 26.0 img/s | [01:16<00:00]
--> Epoch 3/3 Complete | Train Loss: 0.0306, Train Acc: 0.990 | Val Loss: 0.0366, Val Acc: 0.988



## 4. Save checkpoint

In [5]:
ckpt_path = os.path.join(PROJECT_ROOT, "checkpoints/efficientnet_b4_image.pt")
model_path = os.path.join(PROJECT_ROOT, "models/efficientnet_b4_image.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_dim": model.feature_dim,
    "backbone": IMG_CFG["backbone"],
}, ckpt_path)
torch.save({"model_state_dict": model.state_dict(), "feature_dim": model.feature_dim, "backbone": IMG_CFG["backbone"]}, model_path)
print("Saved:", ckpt_path)
print("Saved:", model_path)

Saved: .\checkpoints/efficientnet_b4_image.pt
Saved: .\models/efficientnet_b4_image.pt


## 5. Sanity check — extract F_image for a single sample

This is exactly the function the feature-fusion notebook will call later for every image in the dataset.

In [6]:
model.eval()
sample_img, sample_label = full_dataset[0]
with torch.no_grad():
    F_image = model.extract_features(sample_img.unsqueeze(0).to(device))

print("F_image shape:", tuple(F_image.shape))
print("F_image[0, :10]:", F_image[0, :10].cpu().numpy())

F_image shape: (1, 1792)
F_image[0, :10]: [ 0.01492065 -0.05939911  0.23392907  0.21946537 -0.03608176  0.10147157
 -0.03270831  0.1101504  -0.08765349  0.20769145]


## 6. Feature extraction



In [7]:
import os, sys, time
import torch
from torch.utils.data import DataLoader

features_dir = os.path.join(PROJECT_ROOT, "features")
os.makedirs(features_dir, exist_ok=True)

# DataLoader using config batch_size to keep AMD RX 580 GPU saturated
extract_loader = DataLoader(
    full_dataset,
    batch_size=IMG_CFG["batch_size"],
    shuffle=False,
    num_workers=0,
)

features_dict = {}
model.eval()

total_samples = len(full_dataset)
print(
    f"Extracting 1792-dim F_image features from {total_samples} images on GPU"
    f" ({device})...",
    flush=True,
)
t0 = time.time()
processed = 0

with torch.no_grad():
  for batch_idx, (imgs, labels) in enumerate(extract_loader, 1):
    # Pass image batch to DirectML GPU (AMD Radeon RX 580)
    imgs = imgs.to(device)

    # Extract 1792-dimensional features (F_image)
    F_image_batch = model.extract_features(imgs).cpu()

    start_idx = processed
    for i in range(F_image_batch.size(0)):
      img_idx = start_idx + i
      if not full_dataset.synthetic:
        img_path, label = full_dataset.samples[img_idx]
      else:
        img_path, label = f"synthetic_img_{img_idx}.jpg", int(labels[i])

      features_dict[img_path] = {
          "feature": F_image_batch[i],
          "label": int(labels[i]),
      }

    processed += imgs.size(0)
    elapsed = time.time() - t0
    speed = processed / elapsed if elapsed > 0 else 0
    rem_sec = (total_samples - processed) / speed if speed > 0 else 0
    el_m, el_s = divmod(int(elapsed), 60)
    rem_m, rem_s = divmod(int(rem_sec), 60)
    pct = (processed / total_samples) * 100

    msg = (
        f"\rExtracting Features | Processed {processed}/{total_samples}"
        f" ({pct:.1f}%) | {speed:.1f} img/s | [{el_m:02d}:{el_s:02d}<{rem_m:02d}:{rem_s:02d}]"
    )
    sys.stdout.write(msg)
    sys.stdout.flush()

sys.stdout.write("\n")
sys.stdout.flush()

save_path = os.path.join(features_dir, "image_features.pt")
torch.save(features_dict, save_path)

elapsed = time.time() - t0
file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(
    f"\nSuccess! Extracted 1792-dim features for {len(features_dict)} images in"
    f" {elapsed:.1f}s."
)
print(f"File saved at: {save_path} ({file_size_mb:.2f} MB)")


Extracting 1792-dim F_image features from 10000 images on GPU (privateuseone:0)...
Extracting Features | Processed 10000/10000 (100.0%) | 26.3 img/s | [06:20<00:00]

Success! Extracted 1792-dim features for 10000 images in 380.6s.
File saved at: .\features\image_features.pt (69.63 MB)


## Done — Image branch is ready

You now have:
- A fine-tuned EfficientNet-B4 checkpoint at `checkpoints/efficientnet_b4_image.pt`
- A `model.extract_features(x)` method producing `F_image`

Next: open `02_video_cnn_lstm.ipynb` for the video branch.